# Featuresmith Tutorial: 04 — Detecting Data Leakage with Intelligent Leakage Detection

Master Intelligent Leakage Detection — discovering target correlations, timestamp anomalies, identifier shapes, post-outcome feature names, and duplicate target copies before model training.

---


## 1. The High Cost of Data Leakage
Data leakage is one of the most dangerous bugs in applied machine learning. It occurs when information from the target variable or future state leaks into training features. Models achieve deceptively high validation metrics in development, only to fail completely in production.

### 6 Pattern Detectors in v0.4.0
1. **Target Correlation Detector**: Flags features with Pearson correlation >= 0.99 with the target.
2. **Identifier Shape Detector**: Flags near-unique numeric ID features correlated with the target.
3. **Timestamp Detector**: Identifies future timestamp columns encoding post-outcome information.
4. **Future Information Detector**: Identifies features named like outcome labels (e.g., `refund_date`, `is_cancelled`).
5. **Duplicate Target Detector**: Detects near-identical transformed copies of the target.
6. **Suspicious Correlation Detector**: Flags unexpected strong correlations (>= 0.95).

### Prerequisite: Prepare the Customer Churn Dataset
This notebook loads `examples/data/processed/customer_churn.csv`, which the example scripts generate and which is **not** bundled in the repository. From the repository root, run the two preparation steps first:

```bash
python examples/download_datasets.py  # network fetch (requires scikit-learn)
python examples/prepare_datasets.py
```


### Step 1: Analyze Customer Churn Dataset with Leakage Columns

In [1]:
import os

import featuresmith as fs

data_path = os.path.join("..", "data", "processed", "customer_churn.csv")
dataset = fs.load(data_path)

review_res = fs.review(dataset, target_column="churn_label")

all_findings = [f for s in review_res.sections for f in s.findings]
leakage_findings = [
    f for f in all_findings if "leakage" in f.rule_id or "leakage" in f.title.lower()
]

print(f"Total Leakage Findings Count: {len(leakage_findings)}\n")
for finding in leakage_findings:
    print(f"[{finding.severity.upper()}] Column: {finding.column_name}")
    print(f"  Rule : {finding.rule_id}")
    print(f"  Title: {finding.title}")
    print(f"  Detail: {finding.description}\n")

Total Leakage Findings Count: 4

[CRITICAL] Column: leakage_score
  Rule : leakage.multiple_patterns
  Title: Multiple leakage patterns in column 'leakage_score'
  Detail: Column 'leakage_score' triggered 4 leakage pattern(s): Target Correlation, Identifier Shape, Future Information, Duplicate Target Information. Target Correlation: Column 'leakage_score' correlates with the target 'churn_label' at Pearson 1.000, at or above the leakage threshold of 0.990. Identifier Shape: Column 'leakage_score' has an identifier-like shape (near-unique) and correlates with the target 'churn_label' at Pearson 1.000, at or above the threshold of 0.500. An identifier that tracks the outcome is a common leakage vector. Future Information: Column 'leakage_score' is named like the outcome and correlates with the declared target 'churn_label', which may mean it encodes the outcome. Duplicate Target Information: Column 'leakage_score' correlates with the target 'churn_label' at Pearson 1.000 (threshold 0.999

### Key Takeaways & Connection to Next Tutorial
- Always declare your target column when invoking `fs.review(dataset, target_column=...)`.
- Never deploy a model trained on features triggering `CRITICAL` target leakage findings.

**Next Tutorial**: In `05_dataset_diff.ipynb`, we explore the Dataset Diff Engine (`fs.diff`) to compare dataset snapshot versions.